# Transformer HF/TMF Simulation Suite -- Colab launcher

Runs the project's NGSolve simulation stages (CPU and GPU, DC and AC) by cloning [github.com/seifhamzaoui/hf-simulation](https://github.com/seifhamzaoui/hf-simulation) -- one cell per stage, no desktop GUI needed. Result files (the generated `.mat` matrices) are saved to a Google Drive folder at the end.

**Not included here:** `transformer_geometry.py` / `transformer_geometry_rectangular.py` (the geometry BUILDER scripts). They open Netgen's own 3D viewer window (`from netgen.gui import *` + `Draw(...)`), which needs a real display -- Colab is headless and has none. Build/regenerate geometry locally (or via `simulation_ui.py`'s Geometry Builder screen), commit the resulting `.step` files, and push; this notebook only ever *reads* them.

**GPU cells** need a GPU runtime: `Runtime > Change runtime type > T4 GPU` (or better) before running them.

**Heads up:** `run_capacitance()` and `run_dc_resistance()` (CPU) have a built-in `input("...press Enter to continue")` safety pause before their expensive step -- Colab shows this as a small inline text box under the cell; type anything and hit Enter (or just Enter) to continue. Every other function here runs straight through with no prompt.

## 1. Clone the project from GitHub
If the repo is **private**, plain `git clone` will fail with an authentication error -- generate a Personal Access Token (GitHub Settings > Developer settings > Personal access tokens, `repo` scope) and clone with `https://<TOKEN>@github.com/seifhamzaoui/hf-simulation.git` instead. Paste the token directly into this cell for your own session only; don't commit it anywhere.

In [1]:
import os
import sys

REPO_URL = "https://github.com/seifhamzaoui/hf-simulation.git"
PROJECT_DIR = "/content/hf-simulation"

!git clone {REPO_URL} "{PROJECT_DIR}"

assert os.path.isdir(PROJECT_DIR), f"Clone failed -- check REPO_URL above (private repo needs a token, see markdown above)"
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print("cwd:", os.getcwd())
print("config.py found:", os.path.isfile("config.py"))
print("transformer_model_closed.step found:", os.path.isfile("transformer_model_closed.step"))

Cloning into '/content/hf-simulation'...
remote: Enumerating objects: 77, done.
remote: Counting objects: 100% (77/77), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 77 (delta 22), reused 73 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (77/77), 5.31 MiB | 19.40 MiB/s, done.
Resolving deltas: 100% (22/22), done.
cwd: /content/hf-simulation
config.py found: True
transformer_model_closed.step found: True


## 2. Mount Google Drive (where result files will be saved)
Results are copied here at the end -- nothing is written back to the GitHub repo.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

RESULTS_DRIVE_DIR = "/content/drive/MyDrive/ngsolve_results"  # <-- edit if you want a different folder
os.makedirs(RESULTS_DRIVE_DIR, exist_ok=True)
print("Results will be saved to:", RESULTS_DRIVE_DIR)

Results will be saved to: /content/drive/MyDrive/ngsolve_results


## 3. Install dependencies
NGSolve/Netgen ship proper Linux wheels, so this is a plain `pip install` on Colab (no portable-runtime tricks needed, unlike the Windows .exe packaging).

In [4]:
!pip install -q ngsolve ezdxf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.9/30.9 MB 89.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 157.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 142.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.6/25.6 MB 108.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 148.1 MB/s eta 0:00:0000:01


## 4. CPU Stages -- `simulation_ngsolve.py`
Capacitance (Electrostatics), DC Resistance (DC Conduction), Inductance (curl-curl field solve).

In [ ]:
import simulation_ngsolve as sim

In [ ]:
# Capacitance (Electrostatics) -- saves cap_data.mat
sim.run_capacitance()

In [ ]:
# DC Resistance (DC Conduction) -- saves DCR.mat
sim.run_dc_resistance()

In [ ]:
# Inductance (curl-curl field solve, closed rings) -- saves induc.mat
# Pass test_rings=["ringp1"] for a fast single-ring smoke test instead of the full N-ring matrix.
sim.run_inductance()

## 5. GPU Stages -- `simulation_ngsolve_cuda.py`
**Needs a GPU runtime** (`Runtime > Change runtime type > T4 GPU`) before running this section.

In [ ]:
!pip install -q cupy-cuda12x
!nvidia-smi -L

In [ ]:
import simulation_ngsolve_cuda as simgpu

In [ ]:
# Capacitance -- GPU solve
simgpu.run_capacitance_gpu()

In [ ]:
# DC Resistance -- GPU solve
simgpu.run_dc_resistance_gpu()

In [ ]:
# Inductance (curl-curl field solve, closed rings) -- GPU solve
# Pass test_rings=["ringp1"] for a fast single-ring smoke test instead of the full N-ring matrix.
simgpu.run_inductance_gpu()

In [ ]:
# PEEC self-inductance cross-checks (ringp1 only, free space / with core BEM correction) -- GPU
simgpu.run_inductance_peec_gpu("ringp1")
simgpu.run_inductance_peec_core_gpu("ringp1")

## 6. AC Litz Sweep (CPU) -- `simulation_ngsolve_litz.py`
Full N-ring x frequency-sweep AC inductance/resistance -- far more expensive than the quick single-ring test. Start with the smoke test.

In [ ]:
import simulation_ngsolve_litz as lz

# Quick smoke test (single ring). Call lz.run_litz_sweep() with no args for the FULL sweep
# (all rings x config.sim_frequencies) -- see this function's own docstring for the cost warning first.
lz.run_litz_sweep(test_rings=["ringp1"])

## 7. R/L Ratio Sweep (CPU) -- `simulation_ngsolve_litz_ratio.py`
Cheaper AC/DC ratio sweep from a small representative turn sample, used to scale the full DCR.mat/induc.mat matrices.

In [5]:
import simulation_ngsolve_litz_ratio as lrg

# primary_count/secondary_count default to config.py's LITZ_RATIO_SAMPLE_COUNT_PRIMARY/_SECONDARY.
lrg.run_ratio_sweep()

Representative sample: primary=['ringp1', 'ringp2', 'ringp3', 'ringp4', 'ringp5', 'ringp6', 'ringp7', 'ringp8', 'ringp9', 'ringp10', 'ringp11', 'ringp12', 'ringp13', 'ringp14', 'ringp15', 'ringp16']  secondary=['rings2', 'rings3', 'rings4', 'rings5', 'rings6', 'rings7', 'rings8', 'rings9', 'rings10', 'rings11', 'rings12', 'rings13', 'rings14', 'rings15', 'rings16', 'rings17', 'rings18', 'rings19', 'rings20', 'rings21', 'rings22', 'rings23', 'rings24', 'rings25', 'rings26', 'rings27', 'rings28', 'rings29', 'rings30', 'rings31', 'rings32', 'rings33', 'rings34', 'rings35', 'rings36', 'rings37', 'rings38', 'rings39', 'rings40', 'rings41']
Loading closed-ring geometry...
Refined 4 entrefer-facing core face(s) to maxh=2.0000mm
Meshing (6-ring sample, shared across DC + every frequency)...
mesh: 341636 elements
Building each selected ring's current source...
  1/56 ringp1: current source ready, Rdc_turn=0.000154208 ohm (fill_factor=0.46)
  2/56 ringp2: current source ready, Rdc_turn=0.0001542

{'frequencies_hz': [1000.0,
  2000.0,
  5000.0,
  10000.0,
  20000.0,
  50000.0,
  100000.0,
  200000.0,
  500000.0,
  1000000.0,
  2000000.0,
  2500000.0,
  4000000.0,
  5000000.0,
  6000000.0,
  7000000.0,
  8000000.0,
  9000000.0,
  10000000.0],
 'ratio_R_primary': array([1.00148090e+00, 1.00592360e+00, 1.03702208e+00, 1.14808234e+00,
        1.59223386e+00, 4.69728897e+00, 1.57298751e+01, 5.89910624e+01,
        3.27728423e+02, 9.75946727e+02, 2.03800231e+03, 2.41642485e+03,
        3.31046146e+03, 3.82551735e+03, 4.30791370e+03, 4.76376532e+03,
        5.19612680e+03, 5.60765010e+03, 6.00087570e+03]),
 'ratio_R_secondary': array([1.00094340e+00, 1.00377360e+00, 1.02358477e+00, 1.09433584e+00,
        1.37729172e+00, 3.35581595e+00, 1.03911723e+01, 3.80607555e+01,
        2.12947420e+02, 6.57457636e+02, 1.46159749e+03, 1.76934154e+03,
        2.51988775e+03, 2.95936966e+03, 3.37435768e+03, 3.76951715e+03,
        4.14692058e+03, 4.50829921e+03, 4.85536559e+03]),
 'ratio_L_primary':

## 8. R/L Ratio Sweep (GPU) -- `simulation_ngsolve_litz_ratio_cuda.py`
**Known issue** (from this project's own dev history): the AC (f>0) GPU solve currently fails to converge/factor (ILU+GMRES never got a working preconditioner) -- only the DC (f=0) baseline is confirmed working. Expect a real sweep to error out partway through. Needs the GPU runtime + `cupy` from section 5.

In [ ]:
import simulation_ngsolve_litz_ratio_cuda as lrgpu

lrgpu.run_ratio_sweep_gpu()

## 9. Save result files to Google Drive
Every stage above saves its output `.mat` file into `PROJECT_DIR/ngsolve matrices/` (all of them share that one folder -- `sim.MATRIX_DIR`). This copies just that folder to `RESULTS_DRIVE_DIR` from section 2.

In [ ]:
import shutil
import os

src = os.path.join(PROJECT_DIR, "ngsolve matrices")
shutil.copytree(src, RESULTS_DRIVE_DIR, dirs_exist_ok=True)
print(f"Copied {src} -> {RESULTS_DRIVE_DIR}")
print(os.listdir(RESULTS_DRIVE_DIR))